# 06 - Quality Gate da Camada Gold

Este notebook valida dimensões, fatos e agregados após a execução completa da camada Gold. O objetivo é interromper o pipeline quando forem detectadas violações de unicidade, integridade referencial, regras de negócio ou reconciliação.


In [ ]:
from pyspark.sql import functions as F

CATALOG = "datalake_mvp"
GOLD = "mvp_gold"

def gold_table(nome):
    return spark.table(f"{CATALOG}.{GOLD}.{nome}")

dim_customer = gold_table("dim_customer")
dim_seller = gold_table("dim_seller")
dim_product = gold_table("dim_product")
dim_date = gold_table("dim_date")
dim_order = gold_table("dim_order")
fact_order_items = gold_table("fact_order_items")
fact_payments = gold_table("fact_payments")
agg_vendas_mensais = gold_table("agg_vendas_mensais")
agg_vendas_categoria = gold_table("agg_vendas_categoria")
agg_clientes = gold_table("agg_clientes")
agg_logistica_uf = gold_table("agg_logistica_uf")

validacoes = []

def registrar(teste, condicao_ok, detalhe):
    validacoes.append((teste, "OK" if condicao_ok else "ERRO", detalhe))

print("✓ Tabelas Gold carregadas.")


## 1. Unicidade das chaves

In [ ]:
regras_unicidade = [
    ("dim_customer", dim_customer, ["customer_id"]),
    ("dim_seller", dim_seller, ["seller_id"]),
    ("dim_product", dim_product, ["product_id"]),
    ("dim_date", dim_date, ["date_key"]),
    ("dim_order", dim_order, ["order_id"]),
    ("fact_order_items", fact_order_items, ["order_id", "order_item_id"]),
    ("fact_payments", fact_payments, ["order_id", "payment_sequential"]),
    ("agg_vendas_mensais", agg_vendas_mensais, ["mes_ref"]),
    ("agg_vendas_categoria", agg_vendas_categoria, ["categoria"]),
    ("agg_clientes", agg_clientes, ["customer_unique_id"]),
    ("agg_logistica_uf", agg_logistica_uf, ["uf"])
]

for nome, df, chaves in regras_unicidade:
    total = df.count()
    distintos = df.select(*chaves).distinct().count()
    duplicados = total - distintos
    registrar(f"Unicidade - {nome}", duplicados == 0, f"{duplicados:,} duplicados em {total:,} registros")

print("✓ Unicidade executada.")


## 2. Integridade referencial

In [ ]:
regras_integridade = [
    ("dim_order -> dim_customer", dim_order, dim_customer, ["customer_id"]),
    ("fact_order_items -> dim_customer", fact_order_items, dim_customer, ["customer_id"]),
    ("fact_order_items -> dim_product", fact_order_items, dim_product, ["product_id"]),
    ("fact_order_items -> dim_seller", fact_order_items, dim_seller, ["seller_id"]),
    ("fact_order_items -> dim_date", fact_order_items, dim_date, ["date_key"]),
    ("fact_payments -> dim_customer", fact_payments, dim_customer, ["customer_id"]),
    ("fact_payments -> dim_date", fact_payments, dim_date, ["date_key"])
]

for nome, origem, destino, chaves in regras_integridade:
    orfaos = (
        origem.select(*chaves)
        .filter(F.col(chaves[0]).isNotNull())
        .distinct()
        .join(destino.select(*chaves).distinct(), chaves, "left_anti")
        .count()
    )
    registrar(f"Integridade - {nome}", orfaos == 0, f"{orfaos:,} chaves órfãs")

customer_id_nulo = dim_order.filter(F.col("customer_id").isNull()).count()
registrar("Completude - dim_order.customer_id", customer_id_nulo == 0, f"{customer_id_nulo:,} nulos")
print("✓ Integridade referencial executada.")


## 3. Regras de negócio e consistência financeira

In [ ]:
inconsistencia_atraso = dim_order.filter(
    ((F.col("delivery_delay_days") > 0) & (F.col("flag_late_delivery") != 1)) |
    ((F.col("delivery_delay_days") <= 0) & (F.col("flag_late_delivery") == 1))
).count()
registrar("Regra - coerência flag_late_delivery", inconsistencia_atraso == 0, f"{inconsistencia_atraso:,} inconsistências")

negativos_items = fact_order_items.filter((F.col("price") < 0) | (F.col("freight_value") < 0) | (F.col("item_total_value") < 0)).count()
registrar("Regra - monetários fact_order_items", negativos_items == 0, f"{negativos_items:,} valores negativos")

negativos_payments = fact_payments.filter(F.col("payment_value") < 0).count()
registrar("Regra - monetários fact_payments", negativos_payments == 0, f"{negativos_payments:,} valores negativos")

for nome, df in [("fact_order_items", fact_order_items), ("fact_payments", fact_payments)]:
    nulos = df.filter(F.col("date_key").isNull()).count()
    registrar(f"Completude - {nome}.date_key", nulos == 0, f"{nulos:,} nulos")

inconsistencia_item = fact_order_items.filter(F.abs(F.col("item_total_value") - (F.col("price") + F.col("freight_value"))) > 0.01).count()
registrar("Financeiro - item_total_value", inconsistencia_item == 0, f"{inconsistencia_item:,} divergências")

totais = fact_order_items.agg(F.sum("price").alias("p"), F.sum("freight_value").alias("f"), F.sum("item_total_value").alias("t")).first()
dif = abs((float(totais["p"]) + float(totais["f"])) - float(totais["t"]))
registrar("Financeiro - fechamento fact_order_items", dif <= 0.01, f"Diferença: {dif:.2f}")

payment_nulo = fact_payments.filter(F.col("payment_value").isNull()).count()
registrar("Financeiro - payment_value obrigatório", payment_nulo == 0, f"{payment_nulo:,} nulos")
print("✓ Regras de negócio executadas.")


## 4. Reconciliação dos agregados

In [ ]:
# Mensal x fato
mensal = agg_vendas_mensais.agg(F.sum("itens_vendidos").alias("itens"), F.round(F.sum("valor_total"),2).alias("receita")).first()
fato = fact_order_items.agg(F.count("*").alias("itens"), F.round(F.sum("item_total_value"),2).alias("receita")).first()
registrar("Reconciliação - agg_vendas_mensais itens", mensal["itens"] == fato["itens"], f"agg={mensal['itens']:,} | fato={fato['itens']:,}")
registrar("Reconciliação - agg_vendas_mensais receita", abs(float(mensal["receita"])-float(fato["receita"])) <= 0.01, f"agg={float(mensal['receita']):.2f} | fato={float(fato['receita']):.2f}")

# Categoria x fato
cat = agg_vendas_categoria.agg(F.sum("itens_vendidos").alias("itens"), F.round(F.sum("valor_total"),2).alias("receita")).first()
registrar("Reconciliação - agg_vendas_categoria itens", cat["itens"] == fato["itens"], f"agg={cat['itens']:,} | fato={fato['itens']:,}")
registrar("Reconciliação - agg_vendas_categoria receita", abs(float(cat["receita"])-float(fato["receita"])) <= 0.01, f"agg={float(cat['receita']):.2f} | fato={float(fato['receita']):.2f}")

# Clientes x dimensões/fato
cli = agg_clientes.agg(F.sum("quantidade_pedidos").alias("pedidos"), F.sum("quantidade_itens").alias("itens"), F.round(F.sum("receita_total"),2).alias("receita")).first()
orders_total = dim_order.select("order_id").distinct().count()
registrar("Reconciliação - agg_clientes pedidos", cli["pedidos"] == orders_total, f"agg={cli['pedidos']:,} | dim_order={orders_total:,}")
registrar("Reconciliação - agg_clientes itens", cli["itens"] == fato["itens"], f"agg={cli['itens']:,} | fato={fato['itens']:,}")
registrar("Reconciliação - agg_clientes receita", abs(float(cli["receita"])-float(fato["receita"])) <= 0.01, f"agg={float(cli['receita']):.2f} | fato={float(fato['receita']):.2f}")

# Logística x dim_order
log = agg_logistica_uf.agg(F.sum("pedidos").alias("pedidos"), F.sum("pedidos_entregues").alias("entregues"), F.sum("pedidos_atrasados").alias("atrasados")).first()
ord = dim_order.agg(F.count("*").alias("pedidos"), F.sum(F.when(F.col("order_delivered_customer_date").isNotNull(),1).otherwise(0)).alias("entregues"), F.sum(F.when(F.col("flag_late_delivery")==1,1).otherwise(0)).alias("atrasados")).first()
for campo in ["pedidos", "entregues", "atrasados"]:
    registrar(f"Reconciliação - agg_logistica_uf {campo}", log[campo] == ord[campo], f"agg={log[campo]:,} | dim_order={ord[campo]:,}")

print("✓ Reconciliações executadas.")


## 5. Resultado do Quality Gate

In [ ]:
df_validacoes = spark.createDataFrame(validacoes, ["teste", "status", "detalhe"])
display(df_validacoes)
display(df_validacoes.groupBy("status").count().orderBy("status"))

erros = [(t,d) for t,s,d in validacoes if s == "ERRO"]
print("=" * 60)
print("QUALITY GATE - CAMADA GOLD")
print("=" * 60)
print(f"Testes executados : {len(validacoes)}")
print(f"Testes aprovados  : {len(validacoes)-len(erros)}")
print(f"Testes reprovados : {len(erros)}")

if erros:
    for teste, detalhe in erros:
        print(f"ERRO - {teste}: {detalhe}")
    raise Exception(f"Quality Gate Gold reprovado: {len(erros)} falha(s).")

print("\n✓ QUALITY GATE GOLD APROVADO")
